In [ ]:
#!pip install billboard.py

Note: For this notebook, I found that someone made a Billboard package on GitHub so I tested it out. I'm glad I didn't actually write the code, because they blocked scraping -- I read in an article later that it is now defunct / they don't allow it anymore. But, I do know the json packages, etc. from my previous Python experience, so I know what the code is doing. I then asked ChatGPT to try with last.fm, and also Musicbrainz (you'll see we wrote on the paper that it's open source). But Musicbrainz doesn't list popularity or have valence, etc. so we didn't use it. As you can see, taking the top tracks from Musicbrainz are actually a bunch of random songs and many obscure bands are repeated. It's not reliable.

Since I didn't write the code from scratch on this notebook, I want to be transparent about where it's from / what I did!

Cheers,
Inq

In [2]:
import abc
import json
import unittest
import warnings

import billboard
import six
from billboard import UnsupportedYearWarning

In [ ]:
@six.add_metaclass(abc.ABCMeta)
class Base:
    @classmethod
    @abc.abstractmethod
    def setUpClass(cls):
        pass

    def testYear(self):
        self.assertIsNotNone(self.chart.year)

    def testNextYear(self):
        next_year = str(int(self.chart.year) + 1)
        self.assertEqual(self.chart.nextYear, next_year)

    def testPreviousYear(self):
        previous_year = str(int(self.chart.year) - 1)
        self.assertEqual(self.chart.previousYear, previous_year)

    def testTitle(self):
        self.assertEqual(self.chart.title, self.expectedTitle)

    def testRanks(self):
        ranks = list(entry.rank for entry in self.chart)
        self.assertEqual(ranks, sorted(ranks))
    
    def testEntriesValidity(self, skipTitleCheck=False):
        self.assertEqual(len(self.chart), self.expectedNumEntries)
        for entry in self.chart:
            if not skipTitleCheck:
                self.assertGreater(len(entry.title), 0)
            self.assertGreater(len(entry.artist), 0)

    def testJson(self):
        self.assertTrue(json.loads(self.chart.json()))
        for entry in self.chart:
            self.assertTrue(json.loads(entry.json()))

class TestHot100Songs2019(Base, unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.chart = billboard.ChartData("hot-100-songs", year="2019")
        cls.expectedTitle = "Hot 100 Songs - Year-End"
        cls.expectedNumEntries = 100


class TestHotCountrySongs1970(Base, unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        name = "hot-country-songs"
        year = 1970
        warnings.filterwarnings(action="always", category=UnsupportedYearWarning)
        with warnings.catch_warnings(record=True) as w:
            cls.chart = billboard.ChartData(name, year=year)
            cls.warning = w[0] if w else None

        cls.expectedTitle = "Hot Country Songs - Year-End"
        cls.expectedNumEntries = 100  #just shows the latest chart

    def testUnsupportedYearWarning(self):
        self.assertEquals(self.warning.category, UnsupportedYearWarning)

    def testNextYear(self):
        self.assertIsNone(self.chart.nextYear)

    def testPreviousYear(self):
        self.assertIsNone(self.chart.previousYear)

In [7]:

chart = billboard.ChartData("hot-100-songs", year=1960)


for entry in chart:
    print(f"{entry.rank}. {entry.title} - {entry.artist}")

HTTPError: 403 Client Error: Forbidden for url: https://www.billboard.com/charts/year-end/1960/hot-100-songs

In [ ]:
import requests
import json

API_KEY = "your_lastfm_api_key"
BASE_URL = "http://ws.audioscrobbler.com/2.0/"

def get_top_tracks(year, limit=50):
    """Fetch top tracks for a given year from Last.fm"""
    params = {
        "method": "tag.gettoptracks",
        "tag": str(year),
        "limit": limit,
        "api_key": API_KEY,
        "format": "json"
    }
    
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        tracks = data.get("tracks", {}).get("track", [])
        
        results = []
        for i, track in enumerate(tracks, 1):
            title = track.get("name")
            artist = track.get("artist", {}).get("name")
            results.append(f"{i}. {title} - {artist}")
        
        return results
    else:
        print(f"Error: {response.status_code}")
        return []

top_tracks_1960 = get_top_tracks(1960)

for track in top_tracks_1960:
    print(track)

Error: 403


In [ ]:
import requests

def get_musicbrainz_top_tracks(year, limit=50):
    """Fetch top tracks from MusicBrainz for a given year (ordered by popularity)"""
    url = f"https://musicbrainz.org/ws/2/recording/?query=date:{year}&fmt=json&limit={limit}"
    
    response = requests.get(url, headers={"User-Agent": "YourApp/1.0 ( your@email.com )"})
    
    if response.status_code == 200:
        data = response.json()
        tracks = data.get("recordings", [])
        
        results = []
        for i, track in enumerate(tracks, 1):
            title = track.get("title")
            artist = track.get("artist-credit", [{}])[0].get("name", "Unknown Artist")
            results.append(f"{i}. {title} - {artist}")
        
        return results
    else:
        print(f"Error: {response.status_code}")
        return []

top_tracks_musicbrainz_2010 = get_musicbrainz_top_tracks(2014)

for track in top_tracks_musicbrainz_2010:
    print(track)


1. One More Dub - The Clash
2. You're Like Me Now - Bongwater
3. Celebrity Compass - Bongwater
4. Flop Sweats - Bongwater
5. On the Cusp of 1970 - Bongwater
6. One More Time - The Clash
7. Miss Understanding - L.S.G.
8. Lonely Casseopaya - L.S.G.
9. Lunar Orbit - L.S.G.
10. Reprise - L.S.G.
11. The Leader - The Clash
12. The Crooked Beat - The Clash
13. Lightning Strikes (Not Once but Twice) - The Clash
14. Junco Partner - The Clash
15. The Sound of Sinners - The Clash
16. Flute of Shame - Bongwater
17. Let’s Go Crazy - The Clash
18. Can’t Carry On - Crowded House
19. I Wanted to Boogie - Ten Years After
20. Cryptik Souls Crew - LEN
21. El Condor Pasa (If I Could) - James Galway
22. Perhaps Love - James Galway
23. Poison Years - Bob Mould
24. Heartbreak a Stranger - Bob Mould
25. Things I Wish I’d Said - The Chameleons
26. Nostalgia - The Chameleons
27. Bang Bang - Dispatch
28. Railway - Dispatch
29. Whirlwind - Dispatch
30. Two Coins - Dispatch
31. Pierre in Mist - Brian Eno
32. You K